# 05 — Aggregated Statistics and Figures

**Notebook 5 of 7.** Turns the seven cached eval JSONs (baseline + 3 seeds × 2 conditions) into the numbers and figures that go into the report.

- **Runtime:** ~2 minutes, CPU-only. No GPU needed.
- **Inputs on Drive:** `results/baseline_gsm8k_500.json`, `results/dpo_A_seed{1,2,3}.json`, `results/dpo_B_seed{1,2,3}.json`.
- **Outputs on Drive:**
  - `results/figures/main_results.png` — the headline chart
  - `results/figures/response_length_shift.png` — DPO length-shift audit
  - `results/aggregated_stats.json` — machine-readable summary of every test

Everything here follows the pre-registered protocol in `RESEARCH_PLAN.md` §4:
- **Primary per-seed test:** exact McNemar on discordant pairs (α = 0.05).
- **Pooled test:** McNemar on discordant pairs summed across all three seeds.
- **Effect size:** paired-difference-in-accuracy with 95% percentile bootstrap CI (10,000 resamples).
- **Multiple comparisons:** two condition-vs-baseline families → Bonferroni-adjusted α = 0.025.
- **H2 decision rule:** mean Δ ≥ +3 pp AND pooled McNemar p < 0.025 AND every per-seed 95% CI excludes zero.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/dpo-reasoning-transfer'
os.makedirs(f'{PROJECT_ROOT}/results/figures', exist_ok=True)

CODE_DIR = PROJECT_ROOT   # or f'{PROJECT_ROOT}/repo' if cloned from GitHub
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.stats import (
    load_all_results, check_completeness,
    full_comparison, decide_h2, length_shift_stats,
    per_problem_correctness, per_problem_response_lengths,
)
print('src/stats.py loaded')

## 2. Load results and check completeness

Halts if any of the 7 expected artifacts is missing, with a list of exactly what needs to be run to fill the gap. This is the guardrail against "I only ran two seeds and forgot".

In [ ]:
all_results = load_all_results(PROJECT_ROOT)

warnings = check_completeness(all_results)
if warnings:
    print('CANNOT PROCEED — missing artifacts:')
    for w in warnings:
        print(f'  {w}')
    raise SystemExit('Fix the missing runs, then re-run this notebook.')

print('All expected artifacts present:')
print(f"  baseline: accuracy = {all_results['baseline']['accuracy']:.1%}")
for cond in ('A', 'B'):
    for seed in sorted(all_results[cond]):
        acc = all_results[cond][seed]['accuracy']
        print(f"  Condition {cond}, seed {seed}: accuracy = {acc:.1%}")

## 3. Condition A vs baseline (H1 — the null-result replication)

Under H1 we predicted no significant accuracy change for DPO on general-purpose preferences at this scale.

In [ ]:
cmp_A = full_comparison(all_results['baseline'], all_results['A'], bootstrap_seed=1000)

print('=' * 70)
print(f"CONDITION A (UltraFeedback) vs BASELINE")
print('=' * 70)
print(f"Baseline accuracy:   {cmp_A['baseline_accuracy']:.1%}")
print(f"Mean DPO accuracy:   {cmp_A['mean_dpo_accuracy']:.1%} (sd = {cmp_A['sd_dpo_accuracy']:.3f})")
print(f"Mean delta:          {cmp_A['mean_delta_pp']:+.2f} pp (sd = {cmp_A['sd_delta_pp']:.2f})")
print()
print(f"Per-seed detail:")
for seed, s in cmp_A['per_seed'].items():
    print(f"  seed {seed}: {s['accuracy']:.1%} "
          f"(Δ={s['delta_pp']:+.1f} pp, 95% CI [{s['ci_low_pp']:+.1f}, {s['ci_high_pp']:+.1f}]); "
          f"McNemar b={s['b_right_d_wrong']}, c={s['b_wrong_d_right']}, "
          f"p={s['mcnemar_p']:.4f}")
print()
print(f"Pooled (across all 3 seeds):")
print(f"  discordant: b={cmp_A['pooled_b_right_d_wrong']}, c={cmp_A['pooled_b_wrong_d_right']}")
print(f"  pooled McNemar p = {cmp_A['pooled_mcnemar_p']:.4f}  (threshold after Bonferroni: 0.025)")
print(f"  Cochran's Q = {cmp_A['cochrans_q']:.2f}, p = {cmp_A['cochrans_q_p']:.4f}")

## 4. Condition B vs baseline (H2 — the main hypothesis)

In [ ]:
cmp_B = full_comparison(all_results['baseline'], all_results['B'], bootstrap_seed=2000)

print('=' * 70)
print(f"CONDITION B (self-generated reasoning prefs) vs BASELINE")
print('=' * 70)
print(f"Baseline accuracy:   {cmp_B['baseline_accuracy']:.1%}")
print(f"Mean DPO accuracy:   {cmp_B['mean_dpo_accuracy']:.1%} (sd = {cmp_B['sd_dpo_accuracy']:.3f})")
print(f"Mean delta:          {cmp_B['mean_delta_pp']:+.2f} pp (sd = {cmp_B['sd_delta_pp']:.2f})")
print()
print(f"Per-seed detail:")
for seed, s in cmp_B['per_seed'].items():
    print(f"  seed {seed}: {s['accuracy']:.1%} "
          f"(Δ={s['delta_pp']:+.1f} pp, 95% CI [{s['ci_low_pp']:+.1f}, {s['ci_high_pp']:+.1f}]); "
          f"McNemar b={s['b_right_d_wrong']}, c={s['b_wrong_d_right']}, "
          f"p={s['mcnemar_p']:.4f}")
print()
print(f"Pooled (across all 3 seeds):")
print(f"  discordant: b={cmp_B['pooled_b_right_d_wrong']}, c={cmp_B['pooled_b_wrong_d_right']}")
print(f"  pooled McNemar p = {cmp_B['pooled_mcnemar_p']:.4f}  (threshold after Bonferroni: 0.025)")
print(f"  Cochran's Q = {cmp_B['cochrans_q']:.2f}, p = {cmp_B['cochrans_q_p']:.4f}")

## 5. Pre-registered hypothesis decisions

Applies the decision rules committed to in `RESEARCH_PLAN.md` §4. The rules were fixed before results were seen; running them here is not a choice, just a check.

In [ ]:
print('=' * 70)
print('H1 (Condition A ~ baseline, null result)')
print('=' * 70)
h1_null_supported = (
    abs(cmp_A['mean_delta_pp']) < 2.0
    and cmp_A['pooled_mcnemar_p'] > 0.05
)
print(f"  |mean Δ| < 2 pp:                       "
      f"{abs(cmp_A['mean_delta_pp']):.2f}   "
      f"{'PASS' if abs(cmp_A['mean_delta_pp']) < 2.0 else 'FAIL'}")
print(f"  pooled McNemar p > 0.05:               "
      f"{cmp_A['pooled_mcnemar_p']:.4f}   "
      f"{'PASS' if cmp_A['pooled_mcnemar_p'] > 0.05 else 'FAIL'}")
print(f"  → H1 null-result prediction: {'SUPPORTED' if h1_null_supported else 'NOT SUPPORTED'}")
print()

print('=' * 70)
print('H2 (Condition B > baseline, reasoning-signal DPO helps)')
print('=' * 70)
h2 = decide_h2(cmp_B, alpha_family=0.025)
for reason in h2['reasons']:
    print(f"  {reason}")
print(f"  → H2: {'SUPPORTED' if h2['supported'] else 'NOT SUPPORTED (see failing criterion)'}")

## 6. Main results figure

Bar chart with per-seed dots overlaid. This is the headline figure that goes into the report's abstract.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for the figure
baseline_acc = cmp_A['baseline_accuracy'] * 100
A_accs = [cmp_A['per_seed'][s]['accuracy'] * 100 for s in sorted(cmp_A['per_seed'])]
B_accs = [cmp_B['per_seed'][s]['accuracy'] * 100 for s in sorted(cmp_B['per_seed'])]
A_mean, A_sd = np.mean(A_accs), np.std(A_accs, ddof=1)
B_mean, B_sd = np.mean(B_accs), np.std(B_accs, ddof=1)

fig, ax = plt.subplots(figsize=(7, 5))

xs = [0, 1, 2]
means = [baseline_acc, A_mean, B_mean]
sds = [0, A_sd, B_sd]
colors = ['#7F7F7F', '#4C72B0', '#DD8452']
labels = ['Baseline\n(Instruct)',
          'Condition A\n(UltraFeedback)',
          'Condition B\n(Self-gen reasoning)']

bars = ax.bar(xs, means, yerr=sds, capsize=8, color=colors, alpha=0.85,
              edgecolor='black', linewidth=0.5, error_kw={'lw': 1.2})

# Overlay per-seed dots (jittered slightly)
rng = np.random.default_rng(0)
for x_center, accs in [(1, A_accs), (2, B_accs)]:
    jitter = rng.uniform(-0.08, 0.08, size=len(accs))
    ax.scatter([x_center + j for j in jitter], accs,
               color='black', s=45, zorder=5, edgecolor='white', linewidth=0.8)

# Annotate bar values
for x, m in zip(xs, means):
    ax.text(x, m + 1.5, f'{m:.1f}%', ha='center', va='bottom',
            fontweight='bold', fontsize=11)

# Significance annotation on A and B
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.025: return '*'
    return 'n.s.'

y_top = max(means) + max(sds) + 8
for x, p in [(1, cmp_A['pooled_mcnemar_p']), (2, cmp_B['pooled_mcnemar_p'])]:
    ax.plot([0, x], [y_top, y_top], color='black', lw=1)
    ax.text(x/2, y_top + 0.5, f"{sig_stars(p)}  (p={p:.3g})",
            ha='center', va='bottom', fontsize=9)
    y_top += 4

ax.set_xticks(xs)
ax.set_xticklabels(labels)
ax.set_ylabel('GSM8K[:500] accuracy (%)')
ax.set_title('Effect of DPO preference-signal choice on mathematical reasoning\n'
             'Llama-3.2-3B-Instruct, 500 test problems, 3 seeds per condition',
             fontsize=11)
ax.set_ylim(0, y_top + 8)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# Footer note
ax.text(0.02, 0.98,
        'Bars: mean across seeds  •  error bars: ± sd  •  dots: per-seed  •  ' +
        'p-values: pooled exact McNemar, Bonferroni α=0.025',
        transform=ax.transAxes, fontsize=8, va='top', color='#555555')

plt.tight_layout()
fig_path = f'{PROJECT_ROOT}/results/figures/main_results.png'
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 7. Response-length audit

DPO can inflate or deflate output length. If either condition shifts length substantially, some of the accuracy movement could be attributable to a length effect rather than reasoning quality. This figure and the accompanying stats characterize the shift, feeding directly into the report's limitations discussion.

In [ ]:
# Pool per-problem length deltas across the 3 seeds of each condition
def pool_length_deltas(baseline, condition_dict):
    b_lens = per_problem_response_lengths(baseline)
    deltas = []
    for seed_res in condition_dict.values():
        t_lens = per_problem_response_lengths(seed_res)
        deltas.extend((t_lens - b_lens).tolist())
    return np.array(deltas)

A_deltas = pool_length_deltas(all_results['baseline'], all_results['A'])
B_deltas = pool_length_deltas(all_results['baseline'], all_results['B'])

# Per-seed length shift table
print('Per-seed length shift (chars, vs baseline):')
print()
print(f"{'':25} {'mean Δ':>10} {'median Δ':>10} {'sd Δ':>8} {'longer':>8} {'shorter':>8}")
for cond_name, cond in [('Condition A', all_results['A']), ('Condition B', all_results['B'])]:
    for seed in sorted(cond):
        ls = length_shift_stats(all_results['baseline'], cond[seed])
        print(f"{cond_name} seed {seed}:     {ls['mean_delta']:+10.1f} {ls['median_delta']:+10.1f} "
              f"{ls['sd_delta']:8.1f} {ls['n_longer']:8d} {ls['n_shorter']:8d}")
    print()

# Histogram figure
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, deltas, title, color in [
    (axes[0], A_deltas, 'Condition A (UltraFeedback)', '#4C72B0'),
    (axes[1], B_deltas, 'Condition B (self-gen reasoning)', '#DD8452'),
]:
    ax.hist(deltas, bins=50, color=color, alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='k', ls='--', lw=1, alpha=0.7)
    ax.axvline(deltas.mean(), color='red', ls='-', lw=1.5,
               label=f'mean = {deltas.mean():+.0f} chars')
    ax.set_xlabel('Response length delta vs baseline (chars)')
    ax.set_title(f'{title}\nn = {len(deltas)} (problem × seed)')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Count')

plt.suptitle('DPO response-length shift (pooled over seeds)', fontsize=12, y=1.02)
plt.tight_layout()
fig_path = f'{PROJECT_ROOT}/results/figures/response_length_shift.png'
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 8. Qualitative sample — problems that Condition B flipped correctly

Find problems that Condition B got right in at least 2 of 3 seeds where the baseline got them wrong. These are the "wins" attributable to the reasoning-signal DPO — pick one or two for a qualitative example in the report.

In [ ]:
baseline_correct_ix = set(
    r['index'] for r in all_results['baseline']['results'] if r['correct']
)
baseline_wrong_ix = set(range(all_results['baseline']['n_samples'])) - baseline_correct_ix

# Count how many B seeds got each problem right
B_right_counts = {}
for seed_res in all_results['B'].values():
    for r in seed_res['results']:
        if r['correct']:
            B_right_counts[r['index']] = B_right_counts.get(r['index'], 0) + 1

# Robust wins: baseline wrong, at least 2/3 B seeds right
robust_wins = sorted(
    ix for ix in baseline_wrong_ix
    if B_right_counts.get(ix, 0) >= 2
)
print(f"Robust B wins (baseline wrong, ≥2/3 B seeds right): {len(robust_wins)} problems")
print(f"Problem indices: {robust_wins[:20]}{'...' if len(robust_wins) > 20 else ''}")

# Same for Condition A, for comparison
A_right_counts = {}
for seed_res in all_results['A'].values():
    for r in seed_res['results']:
        if r['correct']:
            A_right_counts[r['index']] = A_right_counts.get(r['index'], 0) + 1
robust_A_wins = sorted(
    ix for ix in baseline_wrong_ix
    if A_right_counts.get(ix, 0) >= 2
)
print(f"Robust A wins (baseline wrong, ≥2/3 A seeds right): {len(robust_A_wins)} problems")

# B-only wins: robust B win AND NOT a robust A win
B_only_wins = sorted(set(robust_wins) - set(robust_A_wins))
print(f"B-only robust wins (B rescued, A did not): {len(B_only_wins)} problems")

# Show 2 examples
print()
print('=' * 70)
print('QUALITATIVE EXAMPLES (for report — B rescued, A did not)')
print('=' * 70)
baseline_by_ix = {r['index']: r for r in all_results['baseline']['results']}
b_seed1_by_ix  = {r['index']: r for r in all_results['B'][1]['results']}

for ix in B_only_wins[:2]:
    b = baseline_by_ix[ix]
    d = b_seed1_by_ix[ix]
    print(f"\n--- Problem #{ix} ---")
    print(f"Q: {b['question']}")
    print(f"Ground truth: {b['gt_answer']}")
    print()
    print(f"BASELINE (wrong, pred = {b['pred_answer']}):")
    print('  ' + b['response'].replace('\n', '\n  ')[:500])
    print()
    print(f"CONDITION B seed 1 (correct, pred = {d['pred_answer']}):")
    print('  ' + d['response'].replace('\n', '\n  ')[:500])

## 9. Save machine-readable summary + copy-paste block for the report

The JSON below is the ground truth for every number that goes into REPORT.md — the write-up should reference it rather than transcribing numbers by hand.

In [ ]:
summary = {
    'baseline': {
        'accuracy': all_results['baseline']['accuracy'],
        'n_samples': all_results['baseline']['n_samples'],
        'n_correct': all_results['baseline']['n_correct'],
    },
    'condition_A_ultrafeedback': cmp_A,
    'condition_B_reasoning': cmp_B,
    'hypotheses': {
        'H1_null_supported': h1_null_supported,
        'H2_supported': h2['supported'],
        'H2_criteria': h2['reasons'],
    },
    'length_shift': {
        'A_pooled': {
            'mean_delta_chars': float(A_deltas.mean()),
            'median_delta_chars': float(np.median(A_deltas)),
            'sd_delta_chars': float(A_deltas.std(ddof=1)),
        },
        'B_pooled': {
            'mean_delta_chars': float(B_deltas.mean()),
            'median_delta_chars': float(np.median(B_deltas)),
            'sd_delta_chars': float(B_deltas.std(ddof=1)),
        },
    },
    'qualitative': {
        'robust_A_wins': len(robust_A_wins),
        'robust_B_wins': len(robust_wins),
        'B_only_robust_wins': len(B_only_wins),
    },
}
summary_path = f'{PROJECT_ROOT}/results/aggregated_stats.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'Saved: {summary_path}')
print()

# Copy-paste block
print('=' * 70)
print('SUMMARY BLOCK — paste into REPORT.md §5 Results')
print('=' * 70)
print()
print(f"Baseline: {all_results['baseline']['accuracy']:.1%} "
      f"({all_results['baseline']['n_correct']}/{all_results['baseline']['n_samples']}) "
      f"on GSM8K[:500].")
print()
print(f"Condition A (UltraFeedback, 500 pairs, 3 seeds):")
print(f"  Mean accuracy: {cmp_A['mean_dpo_accuracy']:.1%} "
      f"± {cmp_A['sd_dpo_accuracy']:.3f}")
print(f"  Mean Δ:        {cmp_A['mean_delta_pp']:+.2f} pp "
      f"± {cmp_A['sd_delta_pp']:.2f}")
print(f"  Pooled McNemar p (Bonferroni α=0.025): {cmp_A['pooled_mcnemar_p']:.4f}")
print(f"  Cochran's Q p: {cmp_A['cochrans_q_p']:.4f}")
print(f"  H1 (null) supported: {h1_null_supported}")
print()
print(f"Condition B (self-generated reasoning, 500 pairs, 3 seeds):")
print(f"  Mean accuracy: {cmp_B['mean_dpo_accuracy']:.1%} "
      f"± {cmp_B['sd_dpo_accuracy']:.3f}")
print(f"  Mean Δ:        {cmp_B['mean_delta_pp']:+.2f} pp "
      f"± {cmp_B['sd_delta_pp']:.2f}")
print(f"  Pooled McNemar p (Bonferroni α=0.025): {cmp_B['pooled_mcnemar_p']:.4f}")
print(f"  Cochran's Q p: {cmp_B['cochrans_q_p']:.4f}")
print(f"  H2 supported: {h2['supported']}")
for r in h2['reasons']:
    print(f"    - {r}")
print()
print(f"Length shift (pooled, chars):")
print(f"  A: mean {A_deltas.mean():+.0f}, sd {A_deltas.std(ddof=1):.0f}")
print(f"  B: mean {B_deltas.mean():+.0f}, sd {B_deltas.std(ddof=1):.0f}")
print()
print(f"Qualitative: {len(B_only_wins)} problems rescued by Condition B "
      f"and NOT by Condition A (≥2/3 seed agreement).")

## Next

- `results/aggregated_stats.json` — the machine-readable summary. Every number in the report should reference this file.
- `results/figures/main_results.png` and `response_length_shift.png` — the two figures for the report body.
- **Next up:** `06_mechanistic_analysis.ipynb` — per-token KL divergence, per-module LoRA weight-norm audit, and qualitative error-type coding on discordant problems. This is what turns the report from "we measured accuracy" into "we measured accuracy AND characterized what changed inside the model.